# Face detection and recognition inference pipeline

The following example illustrates how to use the `facenet_pytorch` python package to perform face detection and recogition on an image dataset using an Inception Resnet V1 pretrained on the VGGFace2 dataset.

The following Pytorch methods are included:
* Datasets
* Dataloaders
* GPU/CPU processing

In [1]:
!pip install facenet-pytorch
from facenet_pytorch import MTCNN, InceptionResnetV1
import torch
from torch.utils.data import DataLoader
from torchvision import datasets
import numpy as np
import pandas as pd

import os

workers = 0 if os.name == 'nt' else 4

#### Determine if an nvidia GPU is available

In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Running on device: {}'.format(device))

Running on device: cpu


#### Define MTCNN module

Default params shown for illustration, but not needed. Note that, since MTCNN is a collection of neural nets and other code, the device must be passed in the following way to enable copying of objects when needed internally.

See `help(MTCNN)` for more details.

In [3]:
mtcnn = MTCNN(
    image_size=160, margin=0, min_face_size=20,
    thresholds=[0.6, 0.7, 0.7], factor=0.709, post_process=True,
    device=device
)

#### Define Inception Resnet V1 module

Set classify=True for pretrained classifier. For this example, we will use the model to output embeddings/CNN features. Note that for inference, it is important to set the model to `eval` mode.

See `help(InceptionResnetV1)` for more details.

In [4]:
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

  0%|          | 0.00/107M [00:00<?, ?B/s]

#### Define a dataset and data loader

We add the `idx_to_class` attribute to the dataset to enable easy recoding of label indices to identity names later one.

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!pip install datasets torch torchvision pillow

import os
import tarfile
from torch.utils.data import DataLoader
import torchvision.datasets as datasets

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Paths to the downloaded zip files in Google Drive
drive_data_dir = "/content/drive/MyDrive/data"
train_zip = os.path.join(drive_data_dir, "vggface2_train.tar.gz")
test_zip = os.path.join(drive_data_dir, "vggface2_test.tar.gz")

# Debugging: List files in the data directory
print(f"Checking directory: {drive_data_dir}")
print("Files in MyDrive/data:")
for f in os.listdir(drive_data_dir):
    if "vggface2" in f.lower():  # Case-insensitive check
        print(f" - {f}")

# Check if files exist
print(f"\nLooking for: {train_zip}")
print(f"Train file exists: {os.path.exists(train_zip)}")
print(f"Looking for: {test_zip}")
print(f"Test file exists: {os.path.exists(test_zip)}")

# Function to extract tar.gz files
def extract_file(file_path, extract_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} not found. Check file name or path in {drive_data_dir}")

    if not os.path.exists(extract_path):
        print(f"Extracting {file_path}...")
        with tarfile.open(file_path, 'r:gz') as tar:
            tar.extractall(path=extract_path)
        print(f"Extracted to {extract_path}")
    else:
        print(f"{extract_path} already exists, skipping extraction")

# Output directories for extracted files (in Colab's local filesystem)
base_dir = '/content/vggface2'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')
os.makedirs(base_dir, exist_ok=True)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking directory: /content/drive/MyDrive/data
Files in MyDrive/data:
 - vggface2_train.tar.gz
 - vggface2_test.tar.gz

Looking for: /content/drive/MyDrive/data/vggface2_train.tar.gz
Train file exists: True
Looking for: /content/drive/MyDrive/data/vggface2_test.tar.gz
Test file exists: True
/content/vggface2/train already exists, skipping extraction
Extracting /content/drive/MyDrive/data/vggface2_test.tar.gz...
Extracted to /content/vggface2/test
Number of training samples: 759002
Number of test samples: 169396
Number of classes: 1


In [5]:
# Extract train and test datasets
extract_file(train_zip, train_dir)
extract_file(test_zip, test_dir)

# Use ImageFolder to load the datasets
train_dataset = datasets.ImageFolder(train_dir)
test_dataset = datasets.ImageFolder(test_dir)

# Create index to class mappings
train_dataset.idx_to_class = {i: c for c, i in train_dataset.class_to_idx.items()}
test_dataset.idx_to_class = {i: c for c, i in test_dataset.class_to_idx.items()}

# Print some info to verify
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")
print(f"Number of classes: {len(train_dataset.classes)}")



/content/vggface2/train already exists, skipping extraction
/content/vggface2/test already exists, skipping extraction
Number of training samples: 759002
Number of test samples: 169396
Number of classes: 1


#### Perfom MTCNN facial detection

Iterate through the DataLoader object and detect faces and associated detection probabilities for each. The `MTCNN` forward method returns images cropped to the detected face, if a face was detected. By default only a single detected face is returned - to have `MTCNN` return all detected faces, set `keep_all=True` when creating the MTCNN object above.

To obtain bounding boxes rather than cropped face images, you can instead call the lower-level `mtcnn.detect()` function. See `help(mtcnn.detect)` for details.

In [13]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install facenet-pytorch, datasets, torch, torchvision, and pillow.
!pip install facenet-pytorch
!pip install datasets torch torchvision pillow

# Import the necessary modules.
from facenet_pytorch import MTCNN, InceptionResnetV1
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import pandas as pd
import os
import tarfile
from google.colab import drive

# Determine the number of workers
workers = 0 if os.name == 'nt' else 4

# Determine if a GPU is available
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Running on device: {}'.format(device))

# Define MTCNN module
mtcnn = MTCNN(
    image_size=160, margin=0, min_face_size=20,
    thresholds=[0.6, 0.7, 0.7], factor=0.709, post_process=True,
    device=device
)

# Define Inception Resnet V1 module
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# Define the collate function to pass images to MTCNN
def collate_fn(batch):
    # Each element of batch is a tuple of (image, label)
    images = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    # Return images and labels directly without converting to tensors.
    return images, torch.tensor(labels)

# Paths to the downloaded zip files in Google Drive
drive_data_dir = "/content/drive/MyDrive/data"
train_zip = os.path.join(drive_data_dir, "vggface2_train.tar.gz")
test_zip = os.path.join(drive_data_dir, "vggface2_test.tar.gz")

# Debugging: List files in the data directory
print(f"Checking directory: {drive_data_dir}")
print("Files in MyDrive/data:")
for f in os.listdir(drive_data_dir):
    if "vggface2" in f.lower():  # Case-insensitive check
        print(f" - {f}")

print(f"\nLooking for: {train_zip}")
print(f"Train file exists: {os.path.exists(train_zip)}")
print(f"Looking for: {test_zip}")
print(f"Test file exists: {os.path.exists(test_zip)}")

def extract_file(file_path, extract_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} not found. Check file name or path in {drive_data_dir}")

    if not os.path.exists(extract_path):
        print(f"Extracting {file_path}...")
        try:
          with tarfile.open(file_path, 'r:gz') as tar:
              tar.extractall(path=extract_path)
          print(f"Extracted to {extract_path}")
        except Exception as e:
          print(f"Error extracting {file_path}: {e}")

    else:
        print(f"{extract_path} already exists, skipping extraction")

base_dir = '/content/vggface2'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')
os.makedirs(base_dir, exist_ok=True)

extract_file(train_zip, train_dir)
extract_file(test_zip, test_dir)

# Ensure data was extracted
if len(os.listdir(train_dir)) == 0:
    raise Exception("No training files were extracted.")
if len(os.listdir(test_dir)) == 0:
    raise Exception("No test files were extracted.")

train_dataset = datasets.ImageFolder(train_dir)
test_dataset = datasets.ImageFolder(test_dir)

train_dataset.idx_to_class = {i: c for c, i in train_dataset.class_to_idx.items()}
test_dataset.idx_to_class = {i: c for c, i in test_dataset.class_to_idx.items()}

workers = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    num_workers=workers,
    shuffle=True,
    collate_fn = collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    num_workers=workers,
    shuffle=True,
    collate_fn = collate_fn
)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")
print(f"Number of classes: {len(train_dataset.classes)}")

aligned_train = []
names_train = []
for batch, labels in train_loader:
    x_aligned_batch = []
    for image in batch:
      x_aligned, probs = mtcnn(image, return_prob=True)
      # Check if any face was detected in the current image
      if x_aligned is not None:
          x_aligned_batch.append(x_aligned)

    # Check if any faces were detected in the entire batch
    if len(x_aligned_batch) > 0:
      aligned_train.extend(x_aligned_batch)
      names_train.extend([train_dataset.idx_to_class[label.item()] for label in labels])
      print(f"Batch processed, faces detected")

aligned_test = []
names_test = []
for batch, labels in test_loader:
    x_aligned_batch = []
    for image in batch:
      x_aligned, probs = mtcnn(image, return_prob=True)
      if x_aligned is not None:
        x_aligned_batch.append(x_aligned)

    if len(x_aligned_batch) > 0:
        aligned_test.extend(x_aligned_batch)
        names_test.extend([test_dataset.idx_to_class[label.item()] for label in labels])
        print(f"Batch processed, faces detected")

if len(aligned_train) > 0:
  aligned = torch.stack(aligned_train).to(device)
  embeddings = resnet(aligned).detach().cpu()

  dists = [[(e1 - e2).norm().item() for e2 in embeddings] for e1 in embeddings]
  print(pd.DataFrame(dists, columns=names_train, index=names_train))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running on device: cuda:0
Checking directory: /content/drive/MyDrive/data
Files in MyDrive/data:
 - vggface2_train.tar.gz
 - vggface2_test.tar.gz

Looking for: /content/drive/MyDrive/data/vggface2_train.tar.gz
Train file exists: True
Looking for: /content/drive/MyDrive/data/vggface2_test.tar.gz
Test file exists: True
/content/vggface2/train already exists, skipping extraction
/content/vggface2/test already exists, skipping extraction
Number of training samples: 759002
Number of test samples: 169396
Number of classes: 1
Batch processed, faces detected
Batch processed, faces detected
Batch processed, faces detected
Batch processed, faces detected
Batch processed, faces detected
Batch processed, faces detected
Batch processed, faces detected
Batch processed, faces detected
Batch processed, faces detected
Batch processed, faces detected
Batch processed, faces det

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e9a39f9b4c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1479, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1443, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.11/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/connection.py", line 948, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/selectors.py", line 415, in select
    fd_event_list = self._selector.poll(timeout)
     

KeyboardInterrupt: 

#### Calculate image embeddings

MTCNN will return images of faces all the same size, enabling easy batch processing with the Resnet recognition module. Here, since we only have a few images, we build a single batch and perform inference on it.

For real datasets, code should be modified to control batch sizes being passed to the Resnet, particularly if being processed on a GPU. For repeated testing, it is best to separate face detection (using MTCNN) from embedding or classification (using InceptionResnetV1), as calculation of cropped faces or bounding boxes can then be performed a single time and detected faces saved for future use.

In [ ]:
aligned = torch.stack(aligned).to(device)
embeddings = resnet(aligned).detach().cpu()

#### Print distance matrix for classes

In [ ]:
dists = [[(e1 - e2).norm().item() for e2 in embeddings] for e1 in embeddings]
print(pd.DataFrame(dists, columns=names, index=names))

                angelina_jolie  bradley_cooper  kate_siegel  paul_rudd  \
angelina_jolie        0.000000        1.344806     0.781201   1.425579   
bradley_cooper        1.344806        0.000000     1.256238   0.922126   
kate_siegel           0.781201        1.256238     0.000000   1.366423   
paul_rudd             1.425579        0.922126     1.366423   0.000000   
shea_whigham          1.448495        0.891145     1.416447   0.985438   

                shea_whigham  
angelina_jolie      1.448495  
bradley_cooper      0.891145  
kate_siegel         1.416447  
paul_rudd           0.985438  
shea_whigham        0.000000  
